In [ ]:
!pip install LughaatNLP



In [15]:
import pandas as pd
from LughaatNLP import LughaatNLP

# Load the dataset from column B (second column)
file_path = 'urdu_stories.xlsx'
data = pd.read_excel(file_path, header=None, usecols=[1])  # Read column B
data.columns = ['story_text']  # Rename for easier access

# Initialize the LughaatNLP object
urdu_text_processing = LughaatNLP()

# Merge all rows into one large string
merged_story = ' '.join(data['story_text'].dropna().astype(str))


In [59]:
# Normalize
text = urdu_text_processing.normalize(merged_story)

In [133]:
def detect_sentence_boundaries(text):
    # Common Urdu sentence-ending punctuation
    end_markers = ['۔', '؟', '!']

    # Initial set of common Urdu sentence-starting words (can be expanded)
    start_words = set([
        "آج", "کل", "پھر", "لیکن", "اگر", "جب", "یہ", "وہ", "ایک",
        "کوئی", "جہاں", "کیونکہ", "اس", "ان", "میں", "ہم", "آپ",
        "تم", "وہاں", "یہاں", "اب", "پہلے", "بعد", "جیسے", "جو",
        "جس", "کیا", "کب", "کہاں", "کیسے", "کتنا", "کون", "چونکہ",
    ])

    sentences = []
    current_sentence = []
    words = text.split()

    for i, word in enumerate(words):
        current_sentence.append(word)
        # Check if the word ends with any of the end markers
        if any(word.endswith(marker) for marker in end_markers):
            sentences.append(' '.join(current_sentence))
            current_sentence = []

            # If the next word exists, add it to start_words
            if i + 1 < len(words):
                next_word = words[i + 1]
                start_words.add(next_word)

    # Add any remaining words as a sentence
    if current_sentence:
        sentences.append(' '.join(current_sentence))

    # Convert start_words to a list
    start_words = list(start_words)


    # Extract end words detected in the text
    end_words_detected = [sentence.split()[-1] for sentence in sentences]

    return sentences, start_words, end_words_detected


In [61]:
len(text)

15725956

In [62]:
# Tokenize the text
tokens = urdu_text_processing.urdu_tokenize(text)

In [63]:
len(tokens)

3740693

In [64]:
print(tokens[:100])

['بھورے', 'والا', 'ملتان،', 'ضلع', 'کا', 'ایک', 'چھوٹا', 'سا', 'قصبہ', 'تھا', 'اور', 'یہی', 'ہمارا', 'ابائی', 'گائوں', 'تھا', '۔', 'ہمارے', 'ابائو', 'اجداد', 'کی', 'یہاں', 'زمینداری', 'تھی', 'اور', 'وہ', 'مدتوں', 'سے', 'یہاں', 'اباد', 'تھے', '۔', 'والد', 'صاحب', 'کو', 'وراثت', 'میں', 'جاگیر', 'تو', 'نہیں', 'ملی،', 'بس', 'تھوڑا', 'سا', 'حصہ', 'ملا', 'تھا،', 'جس', 'کی', 'کاشت', 'سے', 'اچھا', 'گزارہ', 'ہو', 'رہا', 'تھا', '۔', 'تاہم', 'ایک', 'چھوٹاز', 'میندار', 'بھی', 'لگن', 'کے', 'ساتھ', 'اپنی', 'زمینوں', 'کی', 'ابیاری', 'کرتارہے', 'تو', 'نصف', 'عمر', 'گزرنے', 'تک', 'وہ', 'بہت', 'خو', 'شحال', 'ہو', 'جاتا', 'ہے', 'اور', 'کافی', 'رقم', 'پس', 'انداز', 'کر', 'سکتا', 'ہے،', 'بات', 'بس', 'طریقے', 'اور', 'سلیقے', 'سے', 'زندگی', 'گزارنے', 'کی', 'ہے']


In [65]:
import random
from collections import defaultdict, Counter

In [94]:
import nltk
from nltk import FreqDist
from nltk.util import ngrams
from nltk.probability import ConditionalFreqDist


In [95]:
# Generate bigrams and trigrams
bigrams = list(ngrams(tokens, 2))
trigrams = list(ngrams(tokens, 3))


In [107]:
# Conditional Frequency Distribution for bigrams
cfd_bigrams = ConditionalFreqDist((w1, w2) for w1, w2 in bigrams)

# Conditional Frequency Distribution for trigrams
cfd_trigrams = ConditionalFreqDist(((w1, w2), w3) for w1, w2, w3 in trigrams)


In [103]:
from nltk.probability import ConditionalFreqDist, ConditionalProbDist, KneserNeyProbDist

In [112]:
# Apply Witten-Bell smoothing for bigrams
bigram_model = ConditionalProbDist(cfd_bigrams, nltk.WittenBellProbDist, bins=len(set(tokens)))

# Apply Kneser-Ney smoothing for trigrams
trigram_model = ConditionalProbDist(cfd_trigrams, nltk.WittenBellProbDist, bins=len(set(tokens)))

In [138]:
sentences, start_words, end_words = detect_sentence_boundaries(merged_story)

In [142]:
def generate_paragraphs(num_paragraphs=3, num_sentences_range=(5, 10), max_length=25):
    paragraphs = []

    for _ in range(num_paragraphs):
        paragraph = []
        last_sentence_words = []  # To keep track of the last sentence's words

        for _ in range(random.randint(*num_sentences_range)):  # Random number of sentences per paragraph
            # Randomly choose a starting word from start_words or tokens
            start_word = random.choice(start_words) if start_words else random.choice(tokens)
            print(f"Chosen start word: {start_word}")

            # Predict the second word using the bigram model
            if start_word in bigram_model:
                second_word = bigram_model[start_word].generate()
            else:
                second_word = random.choice(tokens)

            sentence = [start_word, second_word]

            # Use trigram model for the third word onwards
            for _ in range(2, max_length):  # Starting from index 2 since first two words are already chosen
                context = tuple(sentence[-2:])  # Last two words form the context for trigrams

                # Predict the next word using the trigram model with backoff to bigram
                if context in trigram_model:
                    next_word = trigram_model[context].generate()
                else:
                    bigram_context = sentence[-1]
                    if bigram_context in bigram_model:
                        next_word = bigram_model[bigram_context].generate()
                    else:
                        next_word = random.choice(tokens)

                # Check if the next word is related to the last sentence but not identical
                if last_sentence_words and next_word in last_sentence_words:
                    # Use a synonym or related word if it's too similar (this is a placeholder)
                    next_word = random.choice([word for word in tokens if word not in last_sentence_words])

                sentence.append(next_word)

                # Stop the sentence if an end word is reached
                if next_word in end_words:
                    break

            # Join the generated words into a single sentence and append to the paragraph
            sentence_text = " ".join(sentence)
            paragraph.append(sentence_text)

            # Update last_sentence_words with the current sentence's words
            last_sentence_words = sentence[-2:]  # Keep the last two words for reference

        # Combine the sentences into a paragraph and add to paragraphs list
        if paragraph:
            paragraphs.append(" ".join(paragraph))

    return paragraphs

In [143]:
paragraphs = generate_paragraphs()

Chosen start word: پہچان
Chosen start word: مجلس
Chosen start word: سمجھائوں
Chosen start word: تائو
Chosen start word: رہے
Chosen start word: شاد
Chosen start word: رقیہ
Chosen start word: پارلر
Chosen start word: ہٹو
Chosen start word: کرسی
Chosen start word: چلچلاتی
Chosen start word: ملال
Chosen start word: ڈرنے
Chosen start word: مہنگا
Chosen start word: بیگلے
Chosen start word: چمن
Chosen start word: مشیّت
Chosen start word: بنگلے
Chosen start word: ابوالبتہ
Chosen start word: اطلاع


In [144]:
for paragraph in paragraphs:
    print(paragraph)
    print("\n")

پہچان لیا تھا ان لوگوں کے نزدیک سخت ناپسندیدہ فعل ہے ۔ مجلس درس پابندیٔ وقت کے ساتھ کھانے کا چانس کم ہونے کی وجہ سے کالین کا بھی موقع ہاتھ سے محروم کردے بھاگ کر اپنے سمجھائوں کیونکر نادان اول تو ویسے ہی انہیں معلوم تھا ۔ تائو جی آپ مجھے مراحم کے ساتھ وہاں جا کر آباد ہونے والی عمومی سی گفتگو کے دوران اکثر فقراء سے میری انکھوں سے ساون رہے ہو ؟ شاد ہوگیا تو ڈون نے کمبل چہرے سے ہاتھ ملاتے ہوئے کہا ۔


رقیہ بی بی جو کچھ کہا تھا ۔ پارلر چلی گئی تھی بہت ہی قیمتی تحفے لائے دلکش ریشمی جوڑے، ٹیپ ریکارڈر، استری اور سنگھار کا بہت سا گزر چکا ہے، حالات بدل ہٹو یہاں سے نہ ملو تو یہ ہے کہ دنیا میرے دم سے ہے تو ان سے پیار اور کہاں کس وقت اور چاہیے ۔ کرسی کو بے وقوف بنادیا مجھ کو دیکھتے ہی فرمایا لمحے میں پہنچ گیا طرف دیکھا کا کوپائلٹ اس کی آنکھیں آنسوئوں سے چمک رہی چلچلاتی دوپہر تھی، مجھے نہیں معلوم وہاں کوئی اس حقیقت سے آگاہ کیا کہ جب شازیہ کا شوہر تم سے ملنے کیلئے ایک رشتہ ہے ملال تو ہوا کا یہ ادنیٰ سا نمونہ ظاہر تھا کہ اس نے بھی خاص توجہ نہیں کی وہ بولیں ۔ ڈرنے اور اس نے جلدی سے سنا دوسرے سے خو